# Build a RAG index

[Module 02](README.md) · [Workshop home](../README.md) · [Endpoint reference](../docs/inference.md)

**20 minutes.** A working retrieval pipeline with no vector database and no framework.

Retrieval augmented generation is four steps, and none of them are magic. Split the
documents, embed the pieces, find the pieces closest to the question, and put those
in the prompt. You will write each one.

The documents in `assets/sample-docs/` describe this workshop's own endpoint, so the
answers you get back are real and checkable.

## Step 1: Connect

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

root = Path.cwd()
if not (root / ".env").is_file() and (root.parent / ".env").is_file():
    root = root.parent
if not (root / ".env").is_file():
    raise RuntimeError("No .env found. See docs/00_prework.md")

load_dotenv(root / ".env", override=True)
client = OpenAI(
    base_url=os.environ["AKAMAI_BASE_URL"].strip().rstrip("/"),
    api_key=os.environ["AKAMAI_API_KEY"].strip(),
    timeout=60.0,
)

EMBED_MODEL = "qwen3-embedding-4b"
CHAT_MODEL = "qwen3-8b"
MODULE_DIR = root / "02_Embeddings_and_RAG"
print("Ready.")

## Step 2: Load the documents

Four markdown files about the models, the limits, tool calling, and embeddings.

In [ ]:
docs_dir = MODULE_DIR / "assets" / "sample-docs"
documents = [(p.name, p.read_text(encoding="utf-8")) for p in sorted(docs_dir.glob("*.md"))]

for name, text in documents:
    print(f"{name:<32} {len(text):>6,} characters")
print(f"\n{len(documents)} documents, {sum(len(t) for _, t in documents):,} characters total")

## Step 3: Split them into chunks

Two reasons to split. The embeddings request has a token budget, and a vector
averaged over a whole document is too vague to match a specific question.

Split on blank lines rather than at a fixed character count, so sentences stay whole.
A chunk cut mid sentence embeds badly. A small overlap carries context across the
boundary, which helps when an answer straddles two chunks.

In [ ]:
def chunk_text(text, max_chars=900, overlap=120):
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks, current = [], ""
    for para in paragraphs:
        if current and len(current) + len(para) + 2 > max_chars:
            chunks.append(current)
            current = (current[-overlap:] + "\n\n" + para) if overlap else para
        else:
            current = f"{current}\n\n{para}" if current else para
    if current:
        chunks.append(current)
    return chunks

chunks, sources = [], []
for name, text in documents:
    for chunk in chunk_text(text):
        chunks.append(chunk)
        sources.append(name)

print(f"{len(chunks)} chunks from {len(documents)} documents")
print(f"average {sum(len(c) for c in chunks) // len(chunks)} characters per chunk\n")
print("--- chunk 0 ---")
print(chunks[0][:300])

## Step 4: Embed the chunks

Send them in small batches to stay inside the per-request token budget. The index is
two lists and one array, kept in the same order, so position `i` in the vectors
matches `chunks[i]` and `sources[i]`.

In [ ]:
import numpy as np

BATCH = 8
vectors = []
for i in range(0, len(chunks), BATCH):
    batch = chunks[i : i + BATCH]
    response = client.embeddings.create(model=EMBED_MODEL, input=batch)
    vectors.extend(item.embedding for item in response.data)
    print(f"  embedded {i + len(batch):>3} / {len(chunks)}")

index = np.array(vectors, dtype=np.float32)
print(f"\nindex shape: {index.shape}  ({index.nbytes / 1024:.0f} KB in memory)")

That is the entire index. No database, no service, no extra dependency. For a
workshop corpus this is not a simplification, it is the right answer. You would reach
for pgvector or similar when the corpus stops fitting in memory or when it has to
survive a restart.

## Step 5: Search

Embed the question with the same model, then score it against every stored vector.
Cosine similarity again, this time vectorised so one line scores the whole index.

In [ ]:
def search(query, top_k=3):
    q = np.array(client.embeddings.create(model=EMBED_MODEL, input=[query]).data[0].embedding,
                 dtype=np.float32)
    denom = np.linalg.norm(index, axis=1) * np.linalg.norm(q)
    denom[denom == 0] = 1e-9
    scores = (index @ q) / denom
    best = np.argsort(scores)[::-1][:top_k]
    return [
        {"text": chunks[i], "source": sources[i], "score": round(float(scores[i]), 4)}
        for i in best
    ]

for hit in search("Which model should I use for tool calling?"):
    print(f"[{hit['score']}] {hit['source']}")
    print(f"      {hit['text'][:150].replace(chr(10), ' ')}...\n")

The top hit should be from `tool-calling.md` or `models.md`. Try your own query
before moving on, and watch the scores. Useful hits usually sit well above the rest.
When every score is close together and mediocre, the corpus probably does not cover
the question, which is the case Step 7 handles.

## Step 6: Answer from the sources

Retrieval returns chunks, not answers. To answer, put the top chunks in the prompt
and tell the model to use only those and to cite them.

The instruction to admit ignorance is not optional. Without it a model will fill the
gap with something plausible.

In [ ]:
SYSTEM = (
    "Answer only from the sources provided. Cite the source filename for each claim. "
    "If the sources do not contain the answer, say exactly: Not in the provided sources."
)

def answer(question, top_k=3):
    hits = search(question, top_k=top_k)
    context = "\n\n".join(f"[{h['source']}]\n{h['text']}" for h in hits)
    reply = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Sources:\n\n{context}\n\nQuestion: {question}"},
        ],
        max_tokens=300,
        temperature=0.0,
    )
    return reply.choices[0].message.content, hits

text, hits = answer("What is the context limit of qwen3-8b, and why might a prompt that fits still be rejected?")
print(text)
print("\nretrieved from:", ", ".join(sorted({h["source"] for h in hits})))

The answer should give 32,768 and explain that the endpoint estimates the token count
itself, so its estimate will not match a local tokenizer. Both facts are in the
corpus. The model did not know them, it read them.

Showing which sources were used is not a nicety. It is the only way a user can check
an answer, and it is the main practical advantage of retrieval over a model answering
from memory.

## Step 7: The question with no answer

The case most people skip, and then ship. The corpus says nothing about pricing.

In [ ]:
text, hits = answer("What does Akamai Cloud Inference charge per million tokens?")
print(text)
print("\nbest score was:", hits[0]["score"], "from", hits[0]["source"])

It should refuse. The retrieval still returned its three best chunks, because
similarity search always returns something, and the top score is probably not much
lower than usual. The refusal comes from the instruction in the system prompt, not
from the scores.

Two things follow. A similarity score cannot tell you whether an answer exists, so do
not use a score threshold as your only guard. And the refusal instruction has to be
explicit, which is why `SYSTEM` names the exact sentence to use.

## Validation

In [ ]:
assert len(chunks) == len(sources) == index.shape[0], "index and chunks are out of sync"
assert index.shape[1] == 2560, f"expected 2560 dimensions, got {index.shape[1]}"

hits = search("Which model accepts image input?")
assert hits[0]["score"] > 0.3, f"top score unexpectedly low: {hits[0]['score']}"

grounded, _ = answer("How many dimensions does the embedding model return?")
assert "2,560" in grounded or "2560" in grounded, f"expected the real number, got: {grounded[:120]}"

print("RAG index complete.")
print(f"{len(chunks)} chunks indexed. Grounded answers work and the refusal path works.")

## Clean up

The index lives in this kernel's memory and disappears when you shut the notebook
down. Nothing was written to disk and nothing was created in your cloud account.

The same functions are available as `workshop_utils/retrieval.py`, so module 04 can
import them instead of making you paste this code again.

## Continue

Next: **[Module 03: Platform Features](../03_Platform_Features/)** for image input,
reasoning mode, and typed JSON output.

Or skip ahead to **[Module 04: Agents](../04_Agents/)**, which turns this search
function into a tool the model decides to call on its own.